# Evaluering av RAG-chatbot

Den här notebooken visar resultatet från `evaluation.py`, där resultatet sparas i `evaluation_results.csv`.

Syftet med evalueringen är att visa:
- hur ofta retrieval hittar relevant information
- hur ofta modellen ger ett användbart svar
- hur den lokala modellen fungerar i praktiken
- vilka styrkor och begränsningar som finns i lösningen


In [11]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

df = pd.read_csv('evaluation_results.csv')
df.head()


,query,ideal_answer,answer_keywords,accepted_doc_refs,retrieval_keywords,answer,top_score,retrieval_top1,retrieval_top3,retrieval_topk,answer_correct,answer_strict_correct,retrieved_count,response_time_seconds,had_error,error_message,k,min_score,provider,model
0,Hur gör jag för att ställa in min iPhone mot f...,Guide 5 beskriver att användaren ska installer...,"company portal, outlook, authenticator, företa...","guide 5, iphone","guide 5, iphone, company portal, outlook, auth...",Följ dessa steg för att ställa in din iPhone m...,0.652960,True,True,True,True,False,3,60.584,False,NaN,3,0.3,ollama,llama3.2:3b
1,Hur gör jag om jag glömt mitt lösenord?,Guide 1 beskriver att användaren ska gå till p...,"portal.nordhamndigital.se/konto, glömt lösenor...","guide 1, lösenord","guide 1, lösenord, återställ lösenord, konto, ...",Om du glömt ditt lösenord kan du följa dessa s...,0.707335,True,True,True,True,True,3,70.348,False,NaN,3,0.3,ollama,llama3.2:3b
2,Hur får jag min skrivare?,Guide 8 beskriver att användaren ska skriva ut...,"followme-nordhamn, followme, passerkort, pin, ...","guide 8, followme","guide 8, followme, skrivare, passerkort, pin, ...",Följ steg 3 i dokumentet: Gå till valfri Follo...,0.724095,True,True,True,True,False,3,38.377,False,NaN,3,0.3,ollama,llama3.2:3b
3,Hur bokar jag mötesrum?,Guide 10 beskriver att användaren ska öppna Ou...,"outlook, kalender, nytt möte, rum, rumslista","guide 10, mötesrum","guide 10, mötesrum, outlook, rumslista, kalend...","För att boka ett mötesrum i Outlook, följ dess...",0.773512,True,True,True,True,True,3,61.321,False,NaN,3,0.3,ollama,llama3.2:3b
4,Jag får felmeddelande när jag loggar in i Citr...,Guide 2 beskriver att användaren ska kontrolle...,"citrix.nordhamndigital.se, citrix workspace, s...","guide 2, citrix","guide 2, citrix, workspace, svart skärm, sessi...",Om du får felmeddelande när du loggar in i Cit...,0.646066,True,True,True,True,False,3,54.510,False,NaN,3,0.3,ollama,llama3.2:3b


## Sammanfattning

Här syns huvudresultatet från körningen med min lokala modell via Ollama.


In [12]:
summary = pd.DataFrame({
    'Metrik': ['Retrieval_top1', 'Retrieval_top3', 'Retrieval_topk', 'Korrekta svar (mjukt)', 'Korrekta svar (strikt)', 'Fel i körning', 'Genomsnittlig svarstid (s)'],
    'Värde': [
        round(df['retrieval_top1'].mean() * 100, 1),
        round(df['retrieval_top3'].mean() * 100, 1),
        round(df['retrieval_topk'].mean() * 100, 1),
        round(df['answer_correct'].mean() * 100, 1),
        round(df['answer_strict_correct'].mean() * 100, 1),
        round(df['had_error'].mean() * 100, 1),
        round(df['response_time_seconds'].mean(), 2),
    ]
})
summary


,Metrik,Värde
0,Retrieval_top1,85.7
1,Retrieval_top3,100.0
2,Retrieval_topk,100.0
3,Korrekta svar (mjukt),100.0
4,Korrekta svar (strikt),28.6
5,Fel i körning,0.0
6,Genomsnittlig svarstid (s),57.7


In [13]:
chart_data = summary[summary['Metrik'] != 'Genomsnittlig svarstid (s)'].copy()
chart = alt.Chart(chart_data).mark_bar().encode(
    x=alt.X(
        'Metrik:N',
        title='Metrik',
        sort=None,
        axis=alt.Axis(labelAngle=0, labelFontSize=15, titleFontSize=16, labelLimit=220)
    ),
    y=alt.Y(
        'Värde:Q',
        title='Andel (%)',
        scale=alt.Scale(domain=[0, 100]),
        axis=alt.Axis(labelFontSize=15, titleFontSize=16)
    ),
    color=alt.Color('Metrik:N', legend=None),
    tooltip=['Metrik', alt.Tooltip('Värde:Q', format='.1f')]
).properties(width=760, height=380, title='Översikt av evalueringen')
chart


alt.Chart(...)

## Tolkning av resultatet

Resultatet visar att retrieval-delen fungerar bra i den här körningen. Relevant information hittas ofta bland de högst rankade chunkarna, och modellen ger i de flesta fall ett svar som kan godkännas i den mjuka bedömningen.

Samtidigt är den strikta träffsäkerheten lägre. Det betyder att modellen ofta ligger nära rätt svar, men inte alltid uttrycker sig exakt så tydligt eller precist som den ideala formuleringen i evalueringen. I praktiken kan användaren ändå få hjälp att hitta rätt del av dokumentet och läsa vidare i källan.

Jag testade initialt även Gemini i min evaluering. Där såg jag att modellen gav mycket snabbare svar och i flera fall också större chans att träffa rätt svar. Jag har ändå valt att gå vidare med Ollama i den här versionen, eftersom min gratisvariant av Gemini tog slut på kvoten och därför inte gav en rättvis eller stabil jämförelse.


## Reflektion kring modellval

Mitt val att fortsätta med Ollama handlar framför allt om att lösningen ska kunna användas lokalt i ett företag utan att känslig information lämnar miljön. Det innebär en tydlig fördel ur säkerhets- och sekretessperspektiv. Nackdelen är att svarstiden blir mycket längre och att modellen inte alltid träffar helt exakt i formuleringen.

Gemini visade däremot att en molnbaserad modell kan ge snabbare svar och ibland bättre träffsäkerhet. För en verklig produkt blir det därför en tydlig avvägning mellan datasekretess, prestanda och svarskvalitet.


## Fråga för fråga

Tabellen nedan visar vilka frågor som fungerade bättre eller sämre, samt modellens svar i förkortad form.


In [14]:
question_view = df[['query', 'top_score', 'retrieved_count', 'response_time_seconds', 'retrieval_top1', 'retrieval_top3', 'answer_correct', 'answer_strict_correct', 'answer']].copy()
question_view['top_score'] = question_view['top_score'].round(3)
question_view['response_time_seconds'] = question_view['response_time_seconds'].round(2)
question_view['Modellens svar (kort)'] = question_view['answer'].astype(str).str.slice(0, 180)
question_view = question_view.drop(columns=['answer'])
question_view


,query,top_score,retrieved_count,response_time_seconds,retrieval_top1,retrieval_top3,answer_correct,answer_strict_correct,Modellens svar (kort)
0,Hur gör jag för att ställa in min iPhone mot f...,0.653,3,60.58,True,True,True,False,Följ dessa steg för att ställa in din iPhone m...
1,Hur gör jag om jag glömt mitt lösenord?,0.707,3,70.35,True,True,True,True,Om du glömt ditt lösenord kan du följa dessa s...
2,Hur får jag min skrivare?,0.724,3,38.38,True,True,True,False,Följ steg 3 i dokumentet: Gå till valfri Follo...
3,Hur bokar jag mötesrum?,0.774,3,61.32,True,True,True,True,"För att boka ett mötesrum i Outlook, följ dess..."
4,Jag får felmeddelande när jag loggar in i Citr...,0.646,3,54.51,True,True,True,False,Om du får felmeddelande när du loggar in i Cit...
5,Hur kommer jag in i min mail?,0.569,3,64.83,False,True,True,False,För att komma in i din e-post kan du följande ...
6,Hur ansluter jag till det trådlösa kontorsnätet?,0.660,3,53.93,True,True,True,False,Följ stegen:\n\n1. Öppna listan över trådlösa ...


## Visualisering per fråga

Det här diagrammet visar vilka frågor som fick träff i retrieval och vilka som också blev godkända svar.


In [15]:
heatmap_data = df[['query', 'retrieval_top1', 'retrieval_top3', 'answer_correct']].copy()
heatmap_data = heatmap_data.rename(columns={
    'query': 'Fråga',
    'retrieval_top1': 'Retrieval top 1',
    'retrieval_top3': 'Retrieval top 3',
    'answer_correct': 'Korrekt svar (mjukt)'
})
heatmap_data = heatmap_data.melt(
    id_vars=['Fråga'],
    var_name='Metrik',
    value_name='Träff'
)
heatmap_data['Träff'] = heatmap_data['Träff'].map({True: 'Ja', False: 'Nej'})

heatmap = alt.Chart(heatmap_data).mark_rect(stroke='white', strokeWidth=1.5).encode(
    y=alt.Y('Fråga:N', title=None, sort=None, axis=alt.Axis(labelFontSize=15, labelLimit=700)),
    x=alt.X('Metrik:N', title='Metrik', axis=alt.Axis(labelAngle=0, labelFontSize=15, titleFontSize=17, labelLimit=220)),
    color=alt.Color(
        'Träff:N',
        scale=alt.Scale(domain=['Ja', 'Nej'], range=['#2E8B57', '#D95F5F']),
        legend=alt.Legend(title='Träff', labelFontSize=15, titleFontSize=16)
    ),
    tooltip=['Fråga', 'Metrik', 'Träff']
).properties(width=520, height=440, title='Resultat per fråga')
heatmap


alt.Chart(...)

## Kort slutsats

Min evaluering visar att den lokala Ollama-modellen nu fungerar bättre tillsammans med den justerade chunkingen. Systemet hittar relevant information, och modellen ger ofta ett svar som ligger tillräckligt nära rätt innehåll för att vara användbart i praktiken.

Den största begränsningen är att svaren inte alltid blir helt exakta enligt den strikta bedömningen. Trots det hjälper lösningen användaren att hitta rätt dokumentdel och ger ett bra stöd för vidare läsning i källan.
